# Light curves

Start with observation times and microlensing parameters, then compute
a light curve with `singlens.magnification`. Separations and source radii
are in Einstein-radius units; times are in days.

This example compares point, uniform-disk, and linearly limb-darkened
sources on the same annual-parallax trajectory.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root))
from example._common import style
style()


In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import singlens as sl


## Specify the event

`AnnualParallax` fixes the source sky position and geocentric reference
epoch. Here times are HJD − 2450000. Keep `t_ref` fixed when fitting `t0`;
the north/east parallax amplitudes remain event parameters.


In [ ]:
times = 8000.0 + jnp.linspace(-40.0, 40.0, 1001)
geometry = sl.AnnualParallax(ra_deg=266.0, dec_deg=-29.0, t_ref=8000.0)
event = dict(t0=8000.0, u0=0.05, t_e=80.0, pi_e_n=0.2, pi_e_e=-0.1)

A_point = sl.magnification(times, **event, parallax=geometry)
A_uniform = sl.magnification(times, **event, rho=0.1, parallax=geometry)
A_linear = sl.magnification(times, **event, rho=0.1, a1=0.5, parallax=geometry)


Omitting `rho` selects a point source. Supplying `rho` averages over a
uniform disk; adding `a1` uses $I(\mu)=1-a_1(1-\mu)$. The output has the
same shape as `times`.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6), layout="constrained")
for label, A in [("Point", A_point), ("Uniform", A_uniform), ("Linear (a₁ = 0.5)", A_linear)]:
    ax.plot(times - 8000.0, A, label=label)
ax.set(xlabel="HJD − 2458000 [day]", ylabel="Magnification", title="Annual parallax")
ax.legend()
plt.show()


Finite source size smooths the peak. Limb darkening changes the relative
contribution of the center and edge of the source, so it changes the
transit shape even at fixed radius. For a static trajectory, omit
`parallax`, `pi_e_n`, and `pi_e_e`. For absolute HJD set `time_origin=0`;
for JD TDB set `use_hjd=False`.

## Repeated evaluation and flux

When evaluating the same source profile and sky geometry repeatedly,
prepare a `Model` and pass a `Parameters` object. A dataset's source and
blend fluxes turn magnification into measured flux through $F=f_s A+f_b$.


In [ ]:
model = sl.Model(source="linear", parallax=geometry)
params = sl.Parameters(**event, rho=0.1, a1=0.5)
evaluate = jax.jit(model.magnification)
A = evaluate(times, params)
flux = model.flux(times, params, f_s=1.2, f_b=0.15)


The first compiled call includes JAX compilation. Later calls can change
event parameters while retaining the same model and time-array shape.

If positions have already been computed, evaluate at real separations or
complex source positions directly:


In [ ]:
A_at_position = model.magnification_at(0.1 + 0.02j, rho=0.1, a1=0.5)
print(float(A_at_position))


Continue with [derivatives](autodiff.ipynb), [inference](inference.ipynb),
or the [API](../api.md). See [numerical accuracy](../numerics.md) before
interpreting differences smaller than the model's numerical error.
